# Data Pipeline — DuckDB Ingestion & Cleaning

**Goal:** Load raw CSVs into a persistent DuckDB database, clean the data, and produce a query-ready analytical layer.  
**Output:** `data/fraud.duckdb` with tables: `raw_transactions`, `raw_identity`, `clean_transactions`, `clean_identity`, and view `merged`

**Why DuckDB?**
- Queries 590K rows in milliseconds vs pandas loading a 652 MB CSV every time
- SQL-native feature engineering (window functions, aggregations)
- Single-file database — easy to share and version


In [ ]:
import duckdb
import pandas as pd
import time
from pathlib import Path

DB_PATH  = '../data/fraud.duckdb'
RAW_PATH = '../data/raw'

# Connect (creates file if it doesn't exist)
con = duckdb.connect(DB_PATH)
print(f'Connected to {DB_PATH}')

## 1. Ingest Raw CSVs

In [ ]:
# --- Ingest train_transaction.csv ---
t0 = time.time()
con.execute(f"""
    CREATE OR REPLACE TABLE raw_transactions AS
    SELECT * FROM read_csv_auto('{RAW_PATH}/train_transaction.csv', header=true)
""")
n_tx = con.execute('SELECT COUNT(*) FROM raw_transactions').fetchone()[0]
print(f'raw_transactions: {n_tx:,} rows  ({time.time()-t0:.1f}s)')

# --- Ingest train_identity.csv ---
t0 = time.time()
con.execute(f"""
    CREATE OR REPLACE TABLE raw_identity AS
    SELECT * FROM read_csv_auto('{RAW_PATH}/train_identity.csv', header=true)
""")
n_id = con.execute('SELECT COUNT(*) FROM raw_identity').fetchone()[0]
print(f'raw_identity:     {n_id:,} rows  ({time.time()-t0:.1f}s)')

In [ ]:
# Sanity check
cols_tx = con.execute('DESCRIBE raw_transactions').df()
cols_id = con.execute('DESCRIBE raw_identity').df()
print(f'raw_transactions columns: {len(cols_tx)}')
print(f'raw_identity columns:     {len(cols_id)}')
print()
print('Sample fraud rate:')
con.execute('SELECT AVG(isFraud) AS fraud_rate, COUNT(*) AS total FROM raw_transactions').df()

## 2. Identify Columns to Drop (>90% Null)

In [ ]:
# Build a SQL query to compute null % for every column
all_cols = cols_tx['column_name'].tolist()

null_exprs = ', '.join([
    f"ROUND(COUNT(*) FILTER (WHERE {c} IS NULL) * 100.0 / COUNT(*), 2) AS \"{c}\""
    for c in all_cols
])
null_df = con.execute(f'SELECT {null_exprs} FROM raw_transactions').df()
null_series = null_df.iloc[0].sort_values(ascending=False)

THRESHOLD = 90.0
cols_to_drop_tx = null_series[null_series > THRESHOLD].index.tolist()
cols_to_keep_tx = [c for c in all_cols if c not in cols_to_drop_tx]

print(f'Columns in raw_transactions:  {len(all_cols)}')
print(f'Columns to DROP (>{THRESHOLD}% null): {len(cols_to_drop_tx)}')
print(f'Columns to KEEP:              {len(cols_to_keep_tx)}')
print()
print('Top 10 worst (being dropped):')
print(null_series.head(10).to_string())

## 3. Create Clean Tables

In [ ]:
# --- clean_transactions ---
# Keep columns under threshold, add engineered datetime fields
BASE_DT_EPOCH = pd.Timestamp('2017-11-30').timestamp()  # reference epoch

keep_cols_sql = ', '.join([f'"{c}"' for c in cols_to_keep_tx])

con.execute(f"""
    CREATE OR REPLACE TABLE clean_transactions AS
    SELECT
        {keep_cols_sql},
        -- Derive datetime from offset (base: 2017-11-30)
        epoch_ms(CAST(TransactionDT + {int(BASE_DT_EPOCH)} AS BIGINT) * 1000) AS transaction_ts,
        hour(epoch_ms(CAST(TransactionDT + {int(BASE_DT_EPOCH)} AS BIGINT) * 1000)) AS hour_of_day,
        dayofweek(epoch_ms(CAST(TransactionDT + {int(BASE_DT_EPOCH)} AS BIGINT) * 1000)) AS day_of_week,
        -- Log-transform amount
        ln(TransactionAmt + 1) AS log_amt,
        -- has_identity flag (will be filled when merging)
        CASE WHEN TransactionID IN (SELECT TransactionID FROM raw_identity) THEN 1 ELSE 0 END AS has_identity,
        -- Encode M-features (T/F -> 1/0)
        CASE WHEN M1 = 'T' THEN 1 WHEN M1 = 'F' THEN 0 ELSE NULL END AS M1_enc,
        CASE WHEN M2 = 'T' THEN 1 WHEN M2 = 'F' THEN 0 ELSE NULL END AS M2_enc,
        CASE WHEN M3 = 'T' THEN 1 WHEN M3 = 'F' THEN 0 ELSE NULL END AS M3_enc,
        CASE WHEN M4 = 'M0' THEN 0 WHEN M4 = 'M1' THEN 1 WHEN M4 = 'M2' THEN 2 ELSE NULL END AS M4_enc,
        CASE WHEN M5 = 'T' THEN 1 WHEN M5 = 'F' THEN 0 ELSE NULL END AS M5_enc,
        CASE WHEN M6 = 'T' THEN 1 WHEN M6 = 'F' THEN 0 ELSE NULL END AS M6_enc,
        CASE WHEN M7 = 'T' THEN 1 WHEN M7 = 'F' THEN 0 ELSE NULL END AS M7_enc,
        CASE WHEN M8 = 'T' THEN 1 WHEN M8 = 'F' THEN 0 ELSE NULL END AS M8_enc,
        CASE WHEN M9 = 'T' THEN 1 WHEN M9 = 'F' THEN 0 ELSE NULL END AS M9_enc
    FROM raw_transactions
""")

n_clean = con.execute('SELECT COUNT(*) FROM clean_transactions').fetchone()[0]
n_cols_clean = len(con.execute('DESCRIBE clean_transactions').df())
print(f'clean_transactions: {n_clean:,} rows x {n_cols_clean} cols')

In [ ]:
# --- clean_identity ---
# Rename columns with hyphens (id-01 -> id_01) and keep all
id_cols = cols_id['column_name'].tolist()

# DuckDB already reads id_XX cleanly; just pass through
con.execute("""
    CREATE OR REPLACE TABLE clean_identity AS
    SELECT * FROM raw_identity
""")

n_id_clean = con.execute('SELECT COUNT(*) FROM clean_identity').fetchone()[0]
print(f'clean_identity: {n_id_clean:,} rows x {len(id_cols)} cols')

## 4. Create Merged View

In [ ]:
# Get identity columns (excluding TransactionID to avoid duplicate)
id_select_cols = [c for c in id_cols if c != 'TransactionID']
id_select_sql = ', '.join([f'i."{c}"' for c in id_select_cols])

con.execute(f"""
    CREATE OR REPLACE VIEW merged AS
    SELECT
        t.*,
        {id_select_sql}
    FROM clean_transactions t
    LEFT JOIN clean_identity i USING (TransactionID)
""")

# Verify
sample = con.execute('SELECT COUNT(*), AVG(isFraud) FROM merged').fetchone()
print(f'merged view: {sample[0]:,} rows, fraud rate = {sample[1]:.2%}')

# Count merged columns
n_merged_cols = len(con.execute('DESCRIBE merged').df())
print(f'Total columns in merged view: {n_merged_cols}')

## 5. Verify Table Summary

In [ ]:
# List all tables and views
print('Tables and views in fraud.duckdb:')
print()
tables = con.execute("""SHOW TABLES""").df()
print(tables.to_string(index=False))
print()

# Quick stats per table
for tbl in ['raw_transactions', 'raw_identity', 'clean_transactions', 'clean_identity']:
    n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    ncols = len(con.execute(f'DESCRIBE {tbl}').df())
    print(f'  {tbl:<25} {n:>8,} rows  x  {ncols} cols')

In [ ]:
# Sample fraud stats from clean layer
print('Fraud rate by ProductCD (from DuckDB):')
con.execute("""
    SELECT
        ProductCD,
        COUNT(*) AS txn_count,
        ROUND(AVG(isFraud)*100, 2) AS fraud_rate_pct
    FROM clean_transactions
    GROUP BY ProductCD
    ORDER BY fraud_rate_pct DESC
""").df()

In [ ]:
# Date range check
print('Date range in clean_transactions:')
con.execute("""
    SELECT
        MIN(transaction_ts) AS earliest,
        MAX(transaction_ts) AS latest,
        COUNT(DISTINCT CAST(transaction_ts AS DATE)) AS distinct_days
    FROM clean_transactions
""").df()

In [ ]:
# Close connection
con.close()

import os
db_size_mb = os.path.getsize(DB_PATH) / 1024**2
print(f'fraud.duckdb size: {db_size_mb:.1f} MB')
print('Done. DuckDB file ready for dbt and model training.')